# Parts 3–4 · 02 Surprise (part 3)

Builds the event list, the point-in-time analyst consensus and SUE from the cache. There is **no WRDS connection** and no dependency on Mahaut's files.

- **Event:** one I/B/E/S quarterly EPS announcement, linked to a CRSP stock.
- **Release time:** the Dow Jones wire time where one exists (from `data/clock_provisional.csv`); otherwise the I/B/E/S time, as a stand-in until EDGAR header times exist. `clock_status` says whether wire and I/B/E/S agree on the session.
- **Consensus:** each analyst's latest estimate for the same quarter, issued and recorded before the release day, within 90 days, from a broker still covering the firm, put on the actual's share basis.
- **SUE:** (actual − median consensus) / price two trading days before the reaction day. There is also a dispersion-scaled version when three or more analysts contribute.

**Needs:** `common.py`, `clock.py`, `surprise.py` in `code/`; `data/trading_calendar.csv` and `data/clock_provisional.csv` (from 00) and `cache/` (from 01).

**Writes:** `data/surprise.parquet`, `data/estimates_used.parquet`, `data/handcheck_sample_50.csv`, `results/funnel_part3.csv`, `results/consensus_counts.csv`.

It loads about 21 million CRSP rows, so the local kernel needs sufficient RAM. If it runs out of memory, use a machine with more RAM.

In [ ]:
import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, SCRIPTS, NOV2_DATA,
                           PROJECT_DIR, STEP5_DIR, PART34_CODE, STEP5_CODE)

# Notebook paths.
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(PART34_CODE))

need = [PART34_CODE / f for f in ["common.py", "clock.py", "surprise.py"]]
need += [PROJECT_DIR / "data" / "trading_calendar.csv", PROJECT_DIR / "cache" / "ibes_actuals"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"missing: {missing}"
print("project folder:", PROJECT_DIR)

## Load the cache

In [ ]:
import importlib, time
import numpy as np
import pandas as pd
import common, clock, surprise as S
for m in (common, clock, S):
    importlib.reload(m)

t0 = time.time()
cal = common.load_trading_calendar(PROJECT_DIR / "data" / "trading_calendar.csv")
act = S.load_cache("ibes_actuals")
link = S.load_cache("link_ibes_crsp")
det = S.load_cache("ibes_detail", columns=["ticker", "estimator", "analys", "pdf", "fpi", "value", "curr",
                                           "report_curr", "fpedats", "anndats", "actdats"])
stops = S.load_cache("ibes_stopped")
names = S.load_cache("crsp_names")
surp = S.load_cache("ibes_surprise_summary")
px = S.load_cache("crsp_daily", columns=["permno", "date", "dlyprc", "dlycumfacshr"])

print(f"actuals {len(act):,} | estimates {len(det):,} | CRSP rows {len(px):,} | "
      f"stopped {'missing' if stops is None else f'{len(stops):,}'} | "
      f"I/B/E/S summary {'missing' if surp is None else f'{len(surp):,}'} | {time.time() - t0:.0f}s")
if stops is None or surp is None:
    print("-> rerun the core pull cell in 01_download: it now also pulls stopped estimates "
          "and the I/B/E/S surprise summary (everything else is skipped)")

## Check the CRSP share factor

The share adjustment relies on CRSP's cumulative share factor jumping by the split ratio. Apple split 7-for-1 on 9 June 2014 and 4-for-1 on 31 August 2020, so both ratios below should print as 7.0 and 4.0.

In [ ]:
aapl = px[px.permno == 14593].set_index("date")["dlycumfacshr"]
for before, after, expected in [("2014-06-06", "2014-06-09", 7), ("2020-08-28", "2020-08-31", 4)]:
    r = aapl.loc[before] / aapl.loc[after]
    print(f"{before} -> {after}: factor ratio {r:.3f} (expected {expected})")

## Events

In [ ]:
funnel = S.Funnel()
ev = S.build_events(act, link, cal, funnel)
print(funnel.frame().to_string(index=False))
print("\nsession types:", ev["session_type"].value_counts().to_dict())

## Wire time first

The instructors' rule: the release time is the Dow Jones wire time; I/B/E/S is the check. `data/clock_provisional.csv` (built in 00) has wire times for about half the events. Where a wire time exists for the same stock within four days, it replaces the I/B/E/S time, which also moves the cut-off for which estimates count. `clock_status` records whether wire and I/B/E/S agree on the session: events where they do are the two-source primary sample.

In [ ]:
clk_path = PROJECT_DIR / "data" / "clock_provisional.csv"
clk = pd.read_csv(clk_path) if clk_path.exists() else None
if clk is None:
    print("no clock_provisional.csv: every event keeps its I/B/E/S time")
ev, wire_counts = S.apply_wire_clock(ev, clk, cal)
for k, v in wire_counts.items():
    print(f"  {k:<46} {v:,}")
print()
print(pd.crosstab(ev["clock_status"], ev["session_type"], margins=True).to_string())

## Point-in-time consensus

In [ ]:
t0 = time.time()
cons, used, counts = S.build_consensus(ev, det, px, stops)
S.check_no_lookahead(used)
for k, v in counts.items():
    print(f"  {k:<52} {v:,}" if isinstance(v, int) else f"  {k:<52} {v}")
print(f"{time.time() - t0:.0f}s")

## SUE

In [ ]:
s = S.add_sue(ev, cons, px, funnel)
f = funnel.frame()
print(f.to_string(index=False))

print("\nanalysts per event:", s["n_analysts"].describe(percentiles=[.1, .5, .9]).round(1).to_dict())
print("share with 3+ analysts:", round(s["n_analysts"].ge(3).mean(), 3))
print("SUE (price-scaled) percentiles:",
      s["sue_price"].quantile([.01, .1, .5, .9, .99]).round(4).to_dict())
u = used[used["event_id"].isin(s["event_id"])]
print("estimates without a CRSP share factor, among events kept:",
      f"{int(u['f_est'].isna().sum()):,} of {len(u):,}")
print("share of positive surprises:", round(s["surprise"].gt(0).mean(), 3),
      "| zero:", round(s["surprise"].eq(0).mean(), 3))

# Preview of the sample filters (applied for real in step 04)
pre = s[s["n_analysts"].ge(3) & s["price_t2"].gt(5)]
print(f"\npreview, 3+ analysts and price above $5: {len(pre):,} events")
print(pre.groupby("split").size().to_dict())
print("of which after Llama 3.1's cutoff (2024-2025):",
      f"{int((pre['reaction_date'] >= '2024-01-01').sum()):,}")
print("\nby clock status (two-source primary sample = wire_ibes_agree):")
print(pd.crosstab(pre["clock_status"], pre["split"], margins=True).to_string())

## Check against I/B/E/S's own surprise figures

Our actual should match theirs almost always. Our consensus won't match exactly, because I/B/E/S uses its own window and filters, but it should be close and agree on the sign of the surprise. The table lists the largest gaps, for a look.

In [ ]:
print("all events:")
gaps = S.compare_with_ibes(s, surp)
print("\n3+ analysts and price above $5:")
S.compare_with_ibes(s[s["n_analysts"].ge(3) & s["price_t2"].gt(5)], surp)
if gaps is not None:
    print("\nlargest gaps, all events:")
    print(gaps.to_string(index=False))

## Save

In [ ]:
(PROJECT_DIR / "results").mkdir(exist_ok=True)
s.to_parquet(PROJECT_DIR / "data" / "surprise.parquet", index=False)
used.to_parquet(PROJECT_DIR / "data" / "estimates_used.parquet", index=False)
f.to_csv(PROJECT_DIR / "results" / "funnel_part3.csv", index=False)
pd.Series({k: v for k, v in counts.items()}).to_csv(PROJECT_DIR / "results" / "consensus_counts.csv", header=["count"])

# The hand-check sample is made once and never redrawn, so work already done
# in it is kept. An older copy only gets the CRSP ticker and name added.
hc_path = PROJECT_DIR / "data" / "handcheck_sample_50.csv"
if not hc_path.exists():
    hc = S.handcheck_sample(s, names=names)
else:
    hc = S.refresh_handcheck(pd.read_csv(hc_path, dtype=str), names)
hc.to_csv(hc_path, index=False)
print(f"saved {len(s):,} events; hand-check sample at {hc_path}")
print(hc[["crsp_ticker", "crsp_name", "ibes_code", "anndats", "actual"]].head(8).to_string(index=False))

In [ ]:
import subprocess
subprocess.run([sys.executable, str(REPO_ROOT / "3_4_surprise_returns" / "code" / "tests" / "run_all.py")], check=True)
